# tensor-to-device — worked example 2: .to(device) — align model and data to the same device

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `tensor-to-device`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

PyTorch raises `RuntimeError` if a model and its input tensors are on different devices. The standard fix is to move both model and every input batch to the same device at the start of the training loop. The idiom `device = 'cuda' if torch.cuda.is_available() else 'cpu'` picks the best available device and is written once, at the top of the script.

## Worked solution

**Step 1 — Pick the device once.**
At the top of the script (or after argument parsing): `device = 'cuda' if t.cuda.is_available() else 'cpu'`. All subsequent `.to(device)` calls use this variable.

**Step 2 — Move the model.**
`model = model.to(device)` (or equivalently `model.to(device)` since `nn.Module.to` is in-place for the module itself, but reassigning is clearer).

**Step 3 — Move each batch.**
Inside the data loop: `X = X.to(device); y = y.to(device)`. Never mutate the dataset tensors — only the batch copies.

**Step 4 — Never mix devices.**
If model is on CUDA and `X` is on CPU, PyTorch raises immediately. Check `x.device == next(model.parameters()).device` when debugging.

In [ ]:
import torch as t
import torch.nn as nn

# Pick device once
device = 'cuda' if t.cuda.is_available() else 'cpu'
print('Using device:', device)

# Model and data
t.manual_seed(0)
model = nn.Linear(6, 2)
model = model.to(device)   # move model

# Simulate a mini-batch
t.manual_seed(0)
X_batch = t.randn(8, 6)   # starts on CPU
y_batch = t.randint(0, 2, (8,))

# Move batch to device before forward pass
X_batch = X_batch.to(device)
y_batch = y_batch.to(device)

print('model device:', next(model.parameters()).device)
print('X_batch device:', X_batch.device)
print('y_batch device:', y_batch.device)

# Forward pass — no device mismatch
logits = model(X_batch)
print('logits shape:', logits.shape)   # (8, 2)